# W1-M1 실습 1, 주파수 예산과 action chunking

lesson.md `§3.2`(5계층 블록도), `§3.3`(대역폭 분리), `§4`(지연 예산 부등식)을 손으로 돌려보는 스크립트입니다.

이 실습에서 확인할 것:

1. 5계층의 동작 주파수를 표로 찍고, 인접 계층 사이 **대역폭 분리비**가 실제로 얼마나 벌어지는지
2. lesson §4.1의 부등식 $H_{chunk} \ge f_2 (T_{replan} + \tau_{infer} + \tau_{comm})$ 을 함수로 구현하고
   lesson의 수치 예(→ 16 스텝)와 퀴즈 5번(→ 41 스텝)을 `assert`로 검증
3. 청크가 짧을 때 생기는 **명령 공백(command gap)** 을 타임라인으로 시뮬레이션
4. $\tau_{infer}$ 스윕 / $H_{chunk}$ 스윕 2-panel 그림 저장
5. **지연 안전 vs 반응성** 트레이드오프를 수치로 (최악 반응 지연 $= H_{chunk}/f_2$)

GPU 불필요. 전체 실행 수 초.

In [ ]:
from __future__ import annotations

import argparse
import math
import sys
import unicodedata
from dataclasses import dataclass
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless 고정. plt.show() 금지, 결과는 전부 파일로

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
from matplotlib import font_manager as fm  # noqa: E402
from matplotlib.ft2font import FT2Font  # noqa: E402

MODULE_ID = "W1-M1"

## 0. 경로와 폰트 유틸

출력은 리포 루트의 `artifacts/W1-M1/`에 저장합니다.
이 파일은 `course/w1-generative-core/01-physical-ai-landscape/practice/` 아래에 있으므로
스크립트로 실행하면 `Path(__file__).resolve().parents[4]`가 리포 루트입니다.
노트북에서는 `__file__`이 없으므로 cwd에서 위로 올라가며 마커 디렉토리를 찾습니다.

In [ ]:
_ROOT_MARKERS = ("course", "docs", "CLAUDE.md")


def find_repo_root() -> Path:
    """리포 루트 디렉토리를 찾는다 (스크립트/노트북 양쪽에서 동작)."""
    try:
        start = Path(__file__).resolve().parent  # .../practice
    except NameError:  # 노트북에는 __file__이 없다
        start = Path.cwd().resolve()
    for cand in (start, *start.parents):
        if all((cand / m).exists() for m in _ROOT_MARKERS):
            return cand
    return start


def artifacts_dir() -> Path:
    out = find_repo_root() / "artifacts" / MODULE_ID
    out.mkdir(parents=True, exist_ok=True)
    return out

In [ ]:
# 한글 폰트 탐색. 집필/실행 환경에 한글 폰트가 없을 수 있으므로
# "이름"이 아니라 "실제 한글 글리프 보유 여부"로 판정하고, 없으면 영문 라벨로 폴백한다.
# (예: 'Noto Sans Gothic'은 이름에 Gothic이 들어가지만 고대 고트 문자용이라 한글이 없다.)
_KO_FONT_PREFERENCE = (
    "Pretendard",
    "NanumGothic",
    "Nanum Gothic",
    "Malgun Gothic",
    "NanumBarunGothic",
    "Noto Sans KR",
    "Noto Sans CJK KR",
    "Noto Sans CJK JP",
    "Source Han Sans KR",
    "AppleGothic",
    "Spoqa Han Sans Neo",
    "UnDotum",
    "Baekmuk Gulim",
)
_PROBE_CHARS = "한글주파수청크"

USE_KOREAN = False  # setup_korean_font()가 갱신


def _has_hangul(font_path: str) -> bool:
    try:
        face = FT2Font(font_path)
        return all(face.get_char_index(ord(c)) != 0 for c in _PROBE_CHARS)
    except Exception:
        return False


def setup_korean_font(force_ascii: bool = False) -> bool:
    """한글 폰트를 찾아 matplotlib 기본 폰트로 설정. 실패하면 영문 폴백."""
    global USE_KOREAN
    plt.rcParams["axes.unicode_minus"] = False  # 마이너스 기호 두부현상 방지
    if force_ascii:
        USE_KOREAN = False
        print("[font] --ascii-labels 지정 → 영문 라벨로 렌더합니다.")
        return False

    installed = {f.name for f in fm.fontManager.ttflist}
    for name in _KO_FONT_PREFERENCE:
        if name not in installed:
            continue
        path = fm.findfont(fm.FontProperties(family=name), fallback_to_default=False)
        if _has_hangul(path):
            plt.rcParams["font.family"] = name
            USE_KOREAN = True
            print(f"[font] 한글 폰트 사용: {name}")
            return True

    USE_KOREAN = False
    print(
        "[font] 경고: 한글 글리프를 가진 폰트를 찾지 못해 그림 라벨을 영문으로 폴백합니다."
        " (해결: apt install fonts-nanum 후 matplotlib 폰트 캐시 삭제)"
    )
    return False


def t(ko: str, en: str) -> str:
    """폰트 상황에 따라 한글/영문 라벨을 고른다."""
    return ko if USE_KOREAN else en

In [ ]:
def _dwidth(s: str) -> int:
    """터미널 표시 폭(한글은 2칸)."""
    return sum(2 if unicodedata.east_asian_width(c) in "WF" else 1 for c in s)


def _pad(s: str, width: int, align: str = "left") -> str:
    gap = max(0, width - _dwidth(s))
    if align == "right":
        return " " * gap + s
    if align == "center":
        left = gap // 2
        return " " * left + s + " " * (gap - left)
    return s + " " * gap


def print_table(headers: list[str], rows: list[list[str]], aligns: list[str] | None = None) -> None:
    """한글 폭을 고려한 간단한 표 출력."""
    aligns = aligns or ["left"] * len(headers)
    widths = [
        max(_dwidth(headers[i]), *(_dwidth(r[i]) for r in rows)) if rows else _dwidth(headers[i])
        for i in range(len(headers))
    ]
    line = "-+-".join("-" * w for w in widths)
    print(" | ".join(_pad(h, w, "center") for h, w in zip(headers, widths)))
    print(line)
    for r in rows:
        print(" | ".join(_pad(c, w, a) for c, w, a in zip(r, widths, aligns)))

## 1. 5계층 주파수 예산 (lesson §3.2)

계층 정의는 lesson §3.2 블록도를 그대로 옮긴 것입니다.
L3(액션 인터페이스)만 고정 주파수가 없습니다. **L4 호출당 1회** 동작하는 이벤트 구동 계층이라서
아래 대역폭 분리 계산에서는 제외합니다.

In [ ]:
@dataclass(frozen=True)
class Layer:
    """스택 한 계층. 주파수는 lesson §3.2 블록도 값."""

    tag: str  # L1 ~ L5
    name_ko: str
    name_en: str
    f_lo: float | None  # Hz, L3는 None(이벤트 구동)
    f_hi: float | None
    io: str  # 입력 → 출력 요약
    company: str  # 회사 스택 요소

    @property
    def f_nominal(self) -> float | None:
        """대표 주파수 = 상·하한의 기하평균 (로그 스케일에서의 중앙값)."""
        if self.f_lo is None or self.f_hi is None:
            return None
        return math.sqrt(self.f_lo * self.f_hi)

    def range_str(self) -> str:
        if self.f_lo is None:
            return "이벤트 구동"
        return f"{self.f_lo:g} ~ {self.f_hi:g} Hz"


# 회사 내비게이션 경로 실측값 (사내 GenP Navigation 모듈 진행 현황, 2026-08-29)
# lesson §3.3 표의 마지막 행 / §7.4 두 경로
NAV2_GLOBAL_HZ = 2.0   # Global Path
NAV2_LOCAL_HZ = 20.0   # Local Path

# lesson §3.2를 따라 위(느리고 똑똑함)에서 아래(빠르고 반사적)로
LAYERS: list[Layer] = [
    Layer("L5", "인지와 매핑", "Perception / Mapping", 0.5, 5.0,
          "RGB-D, LiDAR, 언어목표 → 점군/시맨틱 맵, 목표 3D 좌표",
          "FAST-LIO2 (기하) + FSR-VLN / DualMap (의미)"),
    Layer("L4", "상위 지능(VLA/월드모델)", "High-level (VLA / World Model)", 1.0, 10.0,
          "이미지, 언어, 상태 → 액션 청크 [B,H_chunk,D_action]", "GR00T N1.x VLA / World Model"),
    Layer("L3", "액션 인터페이스", "Action Interface", None, None,
          "액션 의도 → 이산 토큰/setpoint (L4 호출당 1회)", "FSQ 기반 계층 모델"),
    Layer("L2", "전신 제어 WBC", "Whole-Body Control", 50.0, 500.0,
          "상위 명령·관측 이력 → 관절 목표각 q_des [23~43]", "GEAR-SONIC / HOMIE"),
    Layer("L1", "하드웨어·미들웨어", "Hardware / Middleware", 1000.0, 2000.0,
          "q_des → 토크 tau = Kp(q_des-q) - Kd*qdot", "Unitree G1"),
]

# 대역폭 분리를 볼 때는 주파수가 정의된 계층만, 빠른 쪽 → 느린 쪽 순서로 본다.
FREQ_ORDER = ["L1", "L2", "L4", "L5"]

# 캐스케이드 제어의 경험칙: inner loop 대역폭을 outer의 5~10배 (lesson §3.3)
CASCADE_MIN_RATIO = 5.0

In [ ]:
def print_frequency_budget() -> None:
    """lesson §3.2의 5계층 주파수 예산 표를 stdout에 출력."""
    print("\n=== [1] 5계층 주파수 예산 (lesson §3.2) ===")
    rows = []
    for lyr in LAYERS:
        nominal = lyr.f_nominal
        rows.append([
            lyr.tag,
            lyr.name_ko,
            lyr.range_str(),
            f"{nominal:,.1f}" if nominal is not None else "-",
            lyr.company,
        ])
    print_table(
        ["계층", "이름", "동작 주파수", "대표값[Hz]", "회사 스택"],
        rows,
        aligns=["center", "left", "right", "right", "left"],
    )
    print("  * 대표값 = 상·하한의 기하평균. L3는 고정 주파수가 없는 이벤트 구동 계층.")


def separation_ratios() -> list[tuple[str, float, float, float]]:
    """인접 계층(주파수가 정의된 것만)의 대역폭 분리비를 계산.

    반환: (경계 라벨, 빠른 쪽 대표주파수, 느린 쪽 대표주파수, 비율)
    lesson §3.3:  w_L1 >> w_L2 >> w_L4 >> w_L5
    """
    by_tag = {lyr.tag: lyr for lyr in LAYERS}
    out = []
    for fast_tag, slow_tag in zip(FREQ_ORDER[:-1], FREQ_ORDER[1:]):
        f_fast = by_tag[fast_tag].f_nominal
        f_slow = by_tag[slow_tag].f_nominal
        assert f_fast is not None and f_slow is not None
        out.append((f"{fast_tag} / {slow_tag}", f_fast, f_slow, f_fast / f_slow))
    return out


def print_separation_ratios() -> None:
    """lesson §3.3의 w_L1 >> w_L2 >> w_L4 >> w_L5가 수치로 성립하는지 확인."""
    print("\n=== [2] 대역폭 분리비 (lesson §3.3) ===")
    by_tag = {lyr.tag: lyr for lyr in LAYERS}
    rows = []
    for label, f_fast, f_slow, ratio in separation_ratios():
        fast_tag, slow_tag = label.split(" / ")
        # 최선/최악: 밴드가 겹치는지까지 본다 (상한 대 하한 / 하한 대 상한)
        best = by_tag[fast_tag].f_hi / by_tag[slow_tag].f_lo  # type: ignore[operator]
        worst = by_tag[fast_tag].f_lo / by_tag[slow_tag].f_hi  # type: ignore[operator]
        verdict = "OK" if ratio >= CASCADE_MIN_RATIO else "약함"
        rows.append([
            label,
            f"{f_fast:,.1f}",
            f"{f_slow:,.2f}",
            f"x{ratio:,.1f}",
            f"x{best:,.1f}",
            f"x{worst:,.2f}",
            verdict,
        ])
    # 회사 실측: Nav2 지역 20 Hz vs 전역 2 Hz (사내 GenP Navigation 문서, 2026-08-29)
    # 논문 밴드가 아니라 파이프라인에 박힌 설정값이라 최선/최악 범위가 없다.
    nav_ratio = NAV2_LOCAL_HZ / NAV2_GLOBAL_HZ
    rows.append([
        "Nav2 지역/전역 (회사 실측)",
        f"{NAV2_LOCAL_HZ:,.1f}",
        f"{NAV2_GLOBAL_HZ:,.2f}",
        f"x{nav_ratio:,.1f}",
        "범위없음",
        "범위없음",
        "OK" if nav_ratio >= CASCADE_MIN_RATIO else "약함",
    ])
    print_table(
        ["경계", "빠른쪽[Hz]", "느린쪽[Hz]", "대표 비율", "최선", "최악", f"≥x{CASCADE_MIN_RATIO:g}?"],
        rows,
        aligns=["left", "right", "right", "right", "right", "right", "center"],
    )
    print("  읽는 법:")
    print("   - 대표 비율 = 기하평균끼리의 비. 최선 = 빠른쪽 상한 / 느린쪽 하한, 최악 = 빠른쪽 하한 / 느린쪽 상한.")
    print("   - L1/L2, L2/L4는 캐스케이드 경험칙(x5~10)을 넉넉히 만족한다 → 계층 분리가 물리적으로 정당하다.")
    print("   - L4/L5는 대표 비율이 x2.0 수준이고 최악에서는 밴드가 역전(x1 미만)한다.")
    print("     둘 다 '느린 계층'이라 대역폭 분리보다 기능 분리(인지 vs 행동 의도) 성격이 강하다는 뜻이다.")
    print("   - 마지막 행만 성격이 다르다. 앞 세 행은 논문 밴드의 기하평균이지만")
    print(f"     Nav2의 두 값({NAV2_LOCAL_HZ:g} / {NAV2_GLOBAL_HZ:g} Hz)은 우리 파이프라인에 실제로 박힌 설정값이다.")
    print(f"     x{nav_ratio:,.1f}배는 회사가 이미 캐스케이드 경험칙을 만족시키고 있다는 실측이다.")
    print("     lesson §3.3의 '대략 한 자릿수'는 L1~L4 구간에서 성립하고 L4~L5는 그 예외라고 읽으면 된다.")

## 2. 최소 청크 길이 (lesson §4.1 eq.(1))

$$\frac{H_{chunk}}{f_2} \ge T_{replan} + \tau_{infer} + \tau_{comm}
  \quad\Longleftrightarrow\quad
  H_{chunk} \ge f_2\bigl(T_{replan} + \tau_{infer} + \tau_{comm}\bigr),
  \qquad T_{replan} = 1/f_4$$

직관: 청크는 상위 모델의 느림을 감추는 **버퍼**다. 상위가 느릴수록, 하위가 빠를수록 청크가 길어야 한다.
제어 대응: MPC의 예측 지평 $N$과 같은 역할 (receding horizon).

In [ ]:
def min_chunk_length(f2: float, f4: float, tau_infer: float, tau_comm: float) -> int:
    """최소 청크 길이 H_chunk [스텝].

    lesson §4.1 eq.(1):  H_chunk >= f2 * (T_replan + tau_infer + tau_comm),  T_replan = 1/f4

    Args:
        f2: L2(WBC) 제어 주파수 [Hz]
        f4: L4(상위 지능) 재계획 주파수 [Hz]
        tau_infer: L4 추론 지연 [s]
        tau_comm: 통신 지연 [s]

    Returns:
        부등식을 만족하는 최소 정수 스텝 수.
    """
    t_replan = 1.0 / f4                                  # lesson §4.1 eq.(1)
    horizon_s = t_replan + tau_infer + tau_comm          # lesson §4.1 eq.(1) 우변 괄호
    raw = f2 * horizon_s                                 # lesson §4.1 eq.(1)
    # 부동소수 오차로 16.000000000000004 같은 값이 나와 17로 올림되는 것을 막는다.
    return int(math.ceil(raw - 1e-9))


def worst_case_reaction_latency(h_chunk: int, f2: float) -> float:
    """최악 반응 지연 [s] = H_chunk / f2.

    청크 실행 중에는 새 관측을 반영하지 못한다(lesson §4.3 트레이드오프).
    MPC에서 예측 지평 N을 늘릴 때 모델 오차가 누적되는 것과 같은 구조.
    """
    return h_chunk / f2

In [ ]:
def check_lesson_numbers() -> None:
    """lesson의 수치 예를 assert로 검증한다. 여기가 깨지면 코드가 틀린 것."""
    # lesson §4.1 수치 예: f2=50Hz, f4=5Hz(T_replan=200ms), tau_infer=100ms, tau_comm=20ms -> H >= 16
    h = min_chunk_length(f2=50.0, f4=5.0, tau_infer=0.100, tau_comm=0.020)
    assert h == 16, f"lesson §4.1 수치 예 불일치: {h} != 16"

    # lesson 셀프 체크 퀴즈 5번: f2=100Hz, f4=4Hz, tau_infer=150ms, tau_comm=10ms -> H >= 41
    h_quiz = min_chunk_length(f2=100.0, f4=4.0, tau_infer=0.150, tau_comm=0.010)
    assert h_quiz == 41, f"lesson 셀프 체크 퀴즈 5번 불일치: {h_quiz} != 41"

    print("\n=== [3] lesson 수치 검증 ===")
    print(f"  lesson §4.1 예시 f2=50Hz,  f4=5Hz, tau_infer=100ms, tau_comm=20ms -> H >= {h:>3d}  (기대 16) OK")
    print(f"  lesson 퀴즈 5번  f2=100Hz, f4=4Hz, tau_infer=150ms, tau_comm=10ms -> H >= {h_quiz:>3d}  (기대 41) OK")

## 3. 명령 공백(command gap) 시뮬레이션

타이밍 모델은 lesson §4.1의 부등식을 그대로 재현하도록 잡습니다.

- L4는 $t_k = k \cdot T_{replan}$ 에 관측을 찍어 재계획을 건다.
- 그 청크의 액션들은 **관측 시각 $t_k$ 기준으로 시간 인덱싱**되어 있다 (ACT/Diffusion Policy와 동일).
- 청크는 $t_k + \tau_{infer} + \tau_{comm}$ 에 L2에 도착한다.
  → 앞쪽 $\lceil f_2 \tau \rceil$ 개 액션은 이미 지나간 시각용이라 버려진다. **이것이 부등식에 $\tau$가 들어가는 이유다.**
- 따라서 청크 $k$가 실제로 덮는 구간은 $[t_k + \tau,\; t_k + H_{chunk}/f_2)$.

어느 L2 스텝도 이 구간들 중 하나에 들어가지 않으면 **명령 공백**입니다.

In [ ]:
def simulate_command_gap(
    h_chunk: int,
    f2: float = 50.0,
    f4: float = 5.0,
    tau_infer: float = 0.100,
    tau_comm: float = 0.020,
    duration: float = 2.0,
) -> dict:
    """주어진 H_chunk로 L2 타임라인을 돌려 각 스텝의 명령 유무를 판정한다.

    Returns:
        dict: steps(시각 배열), covered(bool 배열), gap_ratio, window_mask 등
    """
    tau = tau_infer + tau_comm
    t_replan = 1.0 / f4
    dt = 1.0 / f2
    eps = 1e-9

    n_steps = int(round(duration * f2))
    steps = np.arange(n_steps) * dt

    n_chunks = int(math.ceil(duration / t_replan)) + 1
    t_obs = np.arange(n_chunks) * t_replan          # 관측 시각 t_k
    starts = t_obs + tau                            # 청크 도착 = 사용 시작
    ends = t_obs + h_chunk / f2                     # 청크가 덮는 마지막 시각(배타)

    covered = np.zeros(n_steps, dtype=bool)
    owner = np.full(n_steps, -1, dtype=int)
    for k, (s, e) in enumerate(zip(starts, ends)):
        if e <= s:  # 청크가 도착 시점에 이미 전부 만료 → 아무것도 못 덮는다
            continue
        m = (steps >= s - eps) & (steps < e - eps)
        covered |= m
        owner[m] = k  # 가장 최근 청크가 우선

    # 첫 청크 도착 전(warm-up)은 정상 부팅 구간이므로 통계에서 제외
    window = steps >= tau - eps
    gap_ratio = float(1.0 - covered[window].mean()) if window.any() else 1.0

    stale_head = int(math.ceil(f2 * tau - 1e-9))  # 도착 시점에 이미 지나간 액션 개수
    return {
        "h_chunk": h_chunk,
        "f2": f2,
        "f4": f4,
        "tau": tau,
        "t_replan": t_replan,
        "steps": steps,
        "covered": covered,
        "owner": owner,
        "window": window,
        "gap_ratio": gap_ratio,
        "stale_head": stale_head,
        "h_required": min_chunk_length(f2, f4, tau_infer, tau_comm),
        "reaction_latency": worst_case_reaction_latency(h_chunk, f2),
    }


def print_gap_report(res: dict, label: str = "", ascii_steps: int = 60) -> None:
    """공백 시뮬 결과를 stdout에 출력 (ASCII 타임라인 포함)."""
    h, f2 = res["h_chunk"], res["f2"]
    print(f"\n=== [4] 명령 공백 시뮬레이션 {label} ===")
    print(
        f"  설정: f2={f2:g}Hz, f4={res['f4']:g}Hz, tau={res['tau'] * 1e3:g}ms, "
        f"T_replan={res['t_replan'] * 1e3:g}ms, H_chunk={h}"
    )
    print(f"  필요 최소 청크 길이 H_required = {res['h_required']} 스텝  (lesson §4.1 eq.(1))")
    print(f"  청크 도착 시 이미 만료된 앞부분 = {res['stale_head']} 스텝  <- tau가 부등식에 들어가는 이유")
    print(f"  명령 공백률 = {res['gap_ratio'] * 100:6.2f} %  ({'정상' if res['gap_ratio'] < 1e-9 else '공백 발생'})")
    print(
        f"  최악 반응 지연 = H_chunk/f2 = {h}/{f2:g} = {res['reaction_latency'] * 1e3:.1f} ms"
        "   <- 청크를 늘린 대가 (lesson §4.3 트레이드오프)"
    )

    n = min(ascii_steps, len(res["steps"]))
    bar = "".join("#" if c else "." for c in res["covered"][:n])
    print(f"  타임라인 첫 {n}스텝 ('#'=명령 있음, '.'=공백):")
    print(f"    |{bar}|")
    print(f"    ^ t=0                                        t={res['steps'][n - 1] * 1e3:.0f}ms")

## 4. 스윕 + 시각화

- 왼쪽 패널: $\tau_{infer}$ 를 10~500 ms로 스윕하며 필요한 $H_{chunk}$.
  오른쪽 보조축에 **최악 반응 지연** $H_{chunk}/f_2$ 를 같이 그려 "지연 안전 vs 반응성"을 한 장에 담는다.
- 오른쪽 패널: $H_{chunk}$ 를 스윕하며 명령 공백률.

In [ ]:
def sweep_tau_infer(
    f2: float, f4: float, tau_comm: float, n_points: int
) -> tuple[np.ndarray, np.ndarray]:
    """tau_infer [10ms, 500ms] 스윕 -> 필요한 H_chunk."""
    taus = np.linspace(0.010, 0.500, n_points)
    hs = np.array([min_chunk_length(f2, f4, float(ti), tau_comm) for ti in taus])
    return taus, hs


def sweep_h_chunk(
    f2: float, f4: float, tau_infer: float, tau_comm: float, h_max: int, duration: float
) -> tuple[np.ndarray, np.ndarray]:
    """H_chunk 스윕 -> 명령 공백률."""
    hs = np.arange(1, h_max + 1)
    gaps = np.array([
        simulate_command_gap(int(h), f2, f4, tau_infer, tau_comm, duration)["gap_ratio"]
        for h in hs
    ])
    return hs, gaps


def plot_budget(
    f2: float,
    f4: float,
    tau_infer: float,
    tau_comm: float,
    n_sweep: int,
    duration: float,
    out_path: Path,
) -> Path:
    """2-panel 그림 저장."""
    h_req = min_chunk_length(f2, f4, tau_infer, tau_comm)

    taus, hs_req = sweep_tau_infer(f2, f4, tau_comm, n_sweep)
    h_max = max(2 * h_req, h_req + 8)
    hs, gaps = sweep_h_chunk(f2, f4, tau_infer, tau_comm, h_max, duration)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.5, 5.2))

    # --- 왼쪽: tau_infer -> 필요한 H_chunk ---
    ax1.plot(taus * 1e3, hs_req, lw=2.2, color="#1f4e79")
    ax1.scatter([tau_infer * 1e3], [h_req], s=140, marker="*", color="#c0392b", zorder=5)
    ax1.annotate(
        t(f"lesson §4.1 예시\n({tau_infer * 1e3:g} ms, H={h_req})",
          f"lesson §4.1 example\n({tau_infer * 1e3:g} ms, H={h_req})"),
        xy=(tau_infer * 1e3, h_req),
        xytext=(tau_infer * 1e3 + 150, h_req - 2.0),
        fontsize=9,
        color="#c0392b",
        arrowprops=dict(arrowstyle="->", color="#c0392b", lw=1.0),
    )
    ax1.set_ylim(bottom=float(hs_req.min()) - 3.0, top=float(hs_req.max()) + 2.0)
    ax1.set_xlabel(t("L4 추론 지연  $\\tau_{infer}$  [ms]", "L4 inference latency  $\\tau_{infer}$  [ms]"))
    ax1.set_ylabel(t("필요한 최소 청크 길이  $H_{chunk}$  [스텝]",
                     "required min chunk length  $H_{chunk}$  [steps]"))
    ax1.set_title(
        t(f"(a) 지연이 커지면 청크도 길어져야 한다   $f_2$={f2:g}Hz, $f_4$={f4:g}Hz",
          f"(a) Longer latency needs longer chunks   $f_2$={f2:g}Hz, $f_4$={f4:g}Hz"),
        fontsize=11,
    )
    ax1.grid(alpha=0.3)
    sec = ax1.secondary_yaxis(
        "right", functions=(lambda h: h / f2 * 1e3, lambda ms: ms * f2 / 1e3)
    )
    sec.set_ylabel(
        t("최악 반응 지연  $H_{chunk}/f_2$  [ms]", "worst-case reaction latency  $H_{chunk}/f_2$  [ms]"),
        color="#7f8c8d",
    )

    # --- 오른쪽: H_chunk -> 공백률 ---
    ax2.step(hs, gaps * 100, where="post", lw=2.2, color="#1f4e79")
    ax2.fill_between(hs, gaps * 100, step="post", alpha=0.15, color="#1f4e79")
    ax2.axvline(h_req, ls="--", lw=1.6, color="#c0392b")
    ax2.annotate(
        t(f"$H_{{required}}$ = {h_req}", f"$H_{{required}}$ = {h_req}"),
        xy=(h_req, 55),
        xytext=(h_req + 1.2, 62),
        fontsize=10,
        color="#c0392b",
    )
    ax2.set_xlabel(t("청크 길이  $H_{chunk}$  [스텝]", "chunk length  $H_{chunk}$  [steps]"))
    ax2.set_ylabel(t("명령 공백률 [%]", "command gap ratio [%]"))
    ax2.set_title(
        t("(b) 청크가 짧으면 L2가 굶는다", "(b) Too-short chunks starve L2"), fontsize=11
    )
    ax2.set_ylim(-3, 100)
    ax2.grid(alpha=0.3)

    fig.suptitle(
        t("W1-M1 · 주파수 예산과 action chunking  (lesson §4.1 eq.(1))",
          "W1-M1 · Frequency budget and action chunking  (lesson §4.1 eq.(1))"),
        fontsize=13,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    fig.savefig(out_path, dpi=140)
    plt.close(fig)
    return out_path

## 5. 실행

In [ ]:
def _in_notebook() -> bool:
    """노트북/커널 안에서 도는가.

    커널의 sys.argv(`-f .../kernel-xxxx.json`)를 argparse에 그대로 넘기면 SystemExit가 난다.
    IPython 유무와 argv[0] 두 가지로 판정한다.
    """
    try:
        from IPython import get_ipython  # type: ignore
        if get_ipython() is not None:
            return True
    except Exception:
        pass
    argv0 = Path(sys.argv[0]).name.lower() if sys.argv else ""
    return argv0 == "" or "ipykernel" in argv0 or "jupyter" in argv0 or "colab" in argv0


def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    p = argparse.ArgumentParser(description="W1-M1: 주파수 예산과 action chunking (lesson §3.2/§3.3/§4)")
    p.add_argument("--smoke", action="store_true", help="스윕 포인트를 줄여 수 초 안에 완주")
    p.add_argument("--f2", type=float, default=50.0, help="L2 제어 주파수 [Hz] (기본 50)")
    p.add_argument("--f4", type=float, default=5.0, help="L4 재계획 주파수 [Hz] (기본 5)")
    p.add_argument("--tau-infer", type=float, default=0.100, help="L4 추론 지연 [s] (기본 0.100)")
    p.add_argument("--tau-comm", type=float, default=0.020, help="통신 지연 [s] (기본 0.020)")
    p.add_argument("--h-chunk", type=int, default=None,
                   help="공백 시뮬에 쓸 청크 길이. 미지정이면 '필요값보다 하나 부족한' 값으로 실패 사례를 보여준다")
    p.add_argument("--duration", type=float, default=2.0, help="시뮬 길이 [s] (기본 2.0)")
    p.add_argument("--ascii-labels", action="store_true", help="한글 폰트가 있어도 영문 라벨로 렌더")
    if argv is None:
        argv = [] if _in_notebook() else sys.argv[1:]
    return p.parse_args(argv)


def main(argv: list[str] | None = None) -> None:
    args = parse_args(argv)
    n_sweep = 8 if args.smoke else 60
    duration = 1.0 if args.smoke else args.duration

    setup_korean_font(force_ascii=args.ascii_labels)

    print("=" * 78)
    print(f" W1-M1 실습 1 — 주파수 예산과 action chunking {'(smoke)' if args.smoke else ''}")
    print("=" * 78)

    print_frequency_budget()
    print_separation_ratios()
    check_lesson_numbers()

    h_req = min_chunk_length(args.f2, args.f4, args.tau_infer, args.tau_comm)

    # 기본 시나리오: 필요값보다 하나 부족하면 어떻게 되는가
    h_used = args.h_chunk if args.h_chunk is not None else max(1, h_req - 1)
    tag = "A: 필요값보다 1스텝 부족" if args.h_chunk is None else f"A: --h-chunk {h_used}"
    print_gap_report(
        simulate_command_gap(h_used, args.f2, args.f4, args.tau_infer, args.tau_comm, duration), tag
    )

    if args.h_chunk is None:
        # 정확히 필요값이면 공백이 사라지는지 확인
        res_ok = simulate_command_gap(h_req, args.f2, args.f4, args.tau_infer, args.tau_comm, duration)
        print_gap_report(res_ok, "B: 정확히 H_required")
        assert res_ok["gap_ratio"] < 1e-9, "H_required에서 공백이 0이어야 한다"

    print("\n=== [5] 지연 안전 vs 반응성 트레이드오프 ===")
    rows = []
    for mult, tag in ((0.5, "부족"), (1.0, "최소(권장)"), (2.0, "여유"), (4.0, "과다")):
        h = max(1, int(round(h_req * mult)))
        g = simulate_command_gap(h, args.f2, args.f4, args.tau_infer, args.tau_comm, duration)
        rows.append([
            f"{h}",
            tag,
            f"{g['gap_ratio'] * 100:.1f}",
            f"{worst_case_reaction_latency(h, args.f2) * 1e3:.0f}",
        ])
    print_table(
        ["H_chunk", "구간", "공백률[%]", "최악 반응지연[ms]"],
        rows,
        aligns=["right", "left", "right", "right"],
    )
    print("  MPC 대응: H_chunk는 receding horizon의 예측 지평 N이다 (lesson §4.3).")
    print("  지평을 늘리면 지연에는 강해지지만 그 구간 동안 새 관측을 반영하지 못한다.")

    # 영문 폴백 렌더는 한글본을 덮어쓰지 않고 따로 저장한다 (02, 03과 동일 규약)
    suffix = "_ascii" if args.ascii_labels else ""
    out = plot_budget(
        args.f2, args.f4, args.tau_infer, args.tau_comm, n_sweep, duration,
        artifacts_dir() / f"01_frequency_budget{suffix}.png",
    )
    print(f"\n[저장] {out}")

In [ ]:
if __name__ == "__main__":
    main()